# 02 - Transform: Mapeamento e Transformação de Dados

## Objetivo
Este notebook é responsável por transformar e enriquecer os dados de exportação através de:
1. Detecção e tratamento de valores vazios
2. Expansão de dados através de dicionários (estados, países, URFs, NCM)
3. Normalização

### Pré-requisitos
Execute primeiro o notebook **01_Extract.ipynb**

## Importações e Configurações

In [ ]:
import pandas as pd
import os

# Configurações
INPUT_DIR = '../data/input'
OUTPUT_DIR = '../data/output'
DICT_DIR = '../data/dictionaries'

# Arquivos
DADOS_CARREGADOS = os.path.join(OUTPUT_DIR, 'dados_carregados.csv')

print("✓ Bibliotecas importadas com sucesso!")

## Carregar Dados da Etapa Anterior

In [ ]:
if os.path.exists(DADOS_CARREGADOS):
    df = pd.read_csv(DADOS_CARREGADOS)
    print(f"✓ Dados carregados: {len(df):,} registros")
    print(f"  Colunas: {len(df.columns)}")
else:
    raise FileNotFoundError(f"Execute primeiro o notebook 01_Extract.ipynb!")

## Funções de Transformação

In [ ]:
def detectar_valores_vazios(df: pd.DataFrame) -> pd.DataFrame:
    """Detecta e preenche valores vazios com zeros."""
    if df.isna().any().any():
        print("Detectados valores vazios, preenchendo com zeros.")
        df = df.fillna(0)
        return df
    else:
        print("Nenhum valor vazio encontrado.")
        return df

def expandir_estados(df: pd.DataFrame, sg_uf_dict_path: str) -> pd.DataFrame:
    """Expande dados de estados através do dicionário."""
    print("Expandindo estados...")
    sg_uf_dict = pd.read_csv(sg_uf_dict_path, sep=";")
    df_merged = pd.merge(df, sg_uf_dict, on="SG_UF_NCM", how="left")
    print(f"  ✓ Estados expandidos: {len(df_merged)} registros")
    return df_merged

def expandir_paises(df: pd.DataFrame, dict_country_path: str) -> pd.DataFrame:
    """Expande dados de países através do dicionário."""
    print("Expandindo países...")
    dict_country = pd.read_csv(dict_country_path, sep=";")
    df_merged = pd.merge(df, dict_country, on="CO_PAIS", how="left")
    print(f"  ✓ Países expandidos: {len(df_merged)} registros")
    return df_merged

def expandir_ncm(df: pd.DataFrame, dict_ncm_path: str) -> pd.DataFrame:
    """Expande dados de produtos NCM através do dicionário."""
    print("Expandindo produtos NCM...")
    dict_ncm = pd.read_csv(dict_ncm_path, sep=";")
    dict_ncm['CO_NCM'] = pd.to_numeric(dict_ncm['CO_NCM'], errors='coerce')
    dict_ncm = dict_ncm.dropna(subset=['CO_NCM'])
    df_merged = pd.merge(df, dict_ncm, on="CO_NCM", how="left")
    print(f"  ✓ NCM expandidos: {len(df_merged)} registros")
    return df_merged

def expandir_urf(df: pd.DataFrame, dict_urf_path: str) -> pd.DataFrame:
    """Expande dados de URFs através do dicionário."""
    print("Expandindo URFs...")
    dict_urf = pd.read_csv(dict_urf_path, sep=";")
    df_merged = pd.merge(df, dict_urf, on="CO_URF", how="left")
    print(f"  ✓ URFs expandidos: {len(df_merged)} registros")
    return df_merged

print("✓ Funções de transformação definidas!")

## Etapa 1: Detectar e Tratar Valores Vazios

In [ ]:
print("=== ETAPA 1: DETECÇÃO DE VALORES VAZIOS ===")
df_original = df.copy()
df = detectar_valores_vazios(df)

# Verificar se houve mudança
valores_nulos_antes = df_original.isnull().sum().sum()
valores_nulos_depois = df.isnull().sum().sum()
print(f"  Valores nulos antes: {valores_nulos_antes:,}")
print(f"  Valores nulos depois: {valores_nulos_depois:,}")

## Etapa 2: Expansão de Dados com Dicionários

In [ ]:
print("=== ETAPA 2: EXPANSÃO DE DADOS ===")
print(f"\nRegistros antes da expansão: {len(df):,}")

# Expandir estados
df = expandir_estados(df, os.path.join(DICT_DIR, 'dict_sg_uf.csv'))

# Expandir países
df = expandir_paises(df, os.path.join(DICT_DIR, 'dict_country.csv'))

# Expandir URFs
df = expandir_urf(df, os.path.join(DICT_DIR, 'dict_urf.csv'))

# Expandir NCM
df = expandir_ncm(df, os.path.join(DICT_DIR, 'dict_ncm_product.csv'))

print(f"\nRegistros após expansão: {len(df):,}")

## Etapa 3: Limpeza Final

In [ ]:
print("=== ETAPA 3: LIMPEZA FINAL ===")

# Remover coluna flag se existir
if "flag" in df.columns:
    df = df.drop(columns=["flag"])
    print("  ✓ Coluna 'flag' removida")

# Verificar resultado final
print(f"  ✓ Total de registros finais: {len(df):,}")
print(f"  ✓ Total de colunas finais: {len(df.columns)}")

## Análise dos Dados Transformados

In [ ]:
print("=== COLUNAS APÓS TRANSFORMAÇÃO ===")
print(df.columns.tolist())

In [ ]:
print("=== AMOSTRA DE DADOS TRANSFORMADOS ===")
df.head()

In [ ]:
# Verificar novas colunas adicionadas
colunas_originais = ['CO_ANO', 'CO_MES', 'CO_NCM', 'CO_UNID', 'CO_PAIS', 'SG_UF_NCM', 'CO_VIA', 'CO_URF', 'QT_ESTAT', 'KG_LIQUIDO', 'VL_FOB']
novas_colunas = [col for col in df.columns if col not in colunas_originais]

print("=== NOVAS COLUNAS ADICIONADAS ===")
for col in novas_colunas:
    print(f"  • {col}")

In [ ]:
# Estatísticas após transformação
print("=== ESTATÍSTICAS APÓS TRANSFORMAÇÃO ===")
df.describe()

## Salvar Dados Transformados

In [ ]:
# Salvar dados transformados
output_file = os.path.join(OUTPUT_DIR, 'dados_transformados.csv')
df.to_csv(output_file, index=False)

print(f"✓ Dados transformados salvos em: {output_file}")
print(f"  Total de registros: {len(df):,}")
print(f"  Total de colunas: {len(df.columns)}")
print(f"  Tamanho do arquivo: {os.path.getsize(output_file)/(1024*1024):.2f} MB")